# Notebook 05 — Final Load Preparation & KPI Framework

**Objective:** Compute KPIs, create dashboard-ready aggregations, and export the final analytical dataset for Tableau visualisation.

**KPI Framework aligned to business problem:** Dealership pricing accuracy, market positioning, and inventory strategy.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

PROCESSED_DIR = '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Libraries loaded. Output directory ready.")

Libraries loaded. Output directory ready.


## 1. Load Cleaned Dataset

In [2]:
CLEANED_PATH = '../data/processed/car_prices_cleaned.csv'

df = pd.read_csv(CLEANED_PATH)

print(f"Cleaned dataset loaded.")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Cleaned dataset loaded.
Shape: 558,675 rows x 29 columns

Columns (29):
['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state', 'condition', 'odometer', 'color', 'interior', 'seller', 'mmr', 'sellingprice', 'saledate', 'sale_year', 'sale_month', 'sale_month_name', 'sale_dow', 'sale_dow_name', 'vehicle_age', 'price_deviation', 'price_deviation_pct', 'price_realization_rate', 'sold_above_mmr', 'price_per_age_year', 'odometer_bucket', 'condition_tier']

First 3 rows:


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate,sale_year,sale_month,sale_month_name,sale_dow,sale_dow_name,vehicle_age,price_deviation,price_deviation_pct,price_realization_rate,sold_above_mmr,price_per_age_year,odometer_bucket,condition_tier
0,2015,Kia,Sorento,Lx,Suv,Automatic,5xyktca69fg566472,ca,5.0000,"16,639.0000",White,Black,Kia Motors America Inc,"20,500.0000","21,500.0000",2014-12-16 12:30:00,"2,014.0000",12.0000,Dec,1.0000,Tue,0.0000,"1,000.0000",4.8800,104.8800,1,"21,500.0000",0–20k,Poor
1,2015,Kia,Sorento,Lx,Suv,Automatic,5xyktca69fg561319,ca,5.0000,"9,393.0000",White,Beige,Kia Motors America Inc,"20,800.0000","21,500.0000",2014-12-16 12:30:00,"2,014.0000",12.0000,Dec,1.0000,Tue,0.0000,700.0000,3.3700,103.3700,1,"21,500.0000",0–20k,Poor
2,2014,Bmw,3 Series,328I Sulev,Sedan,Automatic,wba3c1c51ek116351,ca,45.0000,"1,331.0000",Gray,Black,Financial Services Remarketing (Lease),"31,900.0000","30,000.0000",2015-01-15 04:30:00,"2,015.0000",1.0000,Jan,3.0000,Thu,1.0000,"-1,900.0000",-5.9600,94.0400,0,"30,000.0000",0–20k,Excellent


## 2. KPI Framework

| KPI | Formula | Business Use |
|-----|---------|-------------|
| Avg Price Realization Rate | sellingprice / mmr × 100 | Market competitiveness |
| Price Deviation % | (sellingprice - mmr) / mmr × 100 | Over/under-pricing signal |
| % Sold Above MMR | sold_above_mmr.mean() × 100 | Market demand strength |
| Avg Selling Price | sellingprice.mean() | Portfolio benchmark |
| Total Sales Volume | sellingprice.sum() | Revenue indicator |
| Avg Vehicle Age at Sale | vehicle_age.mean() | Inventory freshness |

In [3]:
# Compute all 6 portfolio-level KPIs
kpi_avg_price_realization = df['price_realization_rate'].mean()
kpi_avg_price_deviation_pct = df['price_deviation_pct'].mean()
kpi_pct_above_mmr = df['sold_above_mmr'].mean() * 100
kpi_avg_selling_price = df['sellingprice'].mean()
kpi_total_sales_volume = df['sellingprice'].sum()
kpi_avg_vehicle_age = df['vehicle_age'].mean()

print("=" * 60)
print("PORTFOLIO-LEVEL KPIs")
print("=" * 60)
print(f"  Avg Price Realization Rate : {kpi_avg_price_realization:.2f}%")
print(f"  Avg Price Deviation %      : {kpi_avg_price_deviation_pct:+.2f}%")
print(f"  % Sold Above MMR           : {kpi_pct_above_mmr:.1f}%")
print(f"  Avg Selling Price          : ${kpi_avg_selling_price:,.2f}")
print(f"  Total Sales Volume         : ${kpi_total_sales_volume:,.0f}")
print(f"  Avg Vehicle Age at Sale    : {kpi_avg_vehicle_age:.1f} years")

# Reference values
print("\n--- Reference Benchmarks ---")
print(f"  Total Transactions         : 558,675")
print(f"  Expected Total Volume      : ~$7.54B")
print(f"  Expected Avg Price         : ~$13,493")
print(f"  Expected % Above MMR       : 46.8%")

PORTFOLIO-LEVEL KPIs
  Avg Price Realization Rate : 98.82%
  Avg Price Deviation %      : -1.18%
  % Sold Above MMR           : 46.8%
  Avg Selling Price          : $13,493.05
  Total Sales Volume         : $7,538,228,609
  Avg Vehicle Age at Sale    : 4.9 years

--- Reference Benchmarks ---
  Total Transactions         : 558,675
  Expected Total Volume      : ~$7.54B
  Expected Avg Price         : ~$13,493
  Expected % Above MMR       : 46.8%


## 3. Make-Level Aggregation (for Tableau — Brand Performance View)

In [4]:
kpi_by_make = df.groupby('make').agg(
    avg_price        = ('sellingprice', 'mean'),
    avg_mmr          = ('mmr', 'mean'),
    avg_deviation_pct= ('price_deviation_pct', 'mean'),
    transaction_count= ('sellingprice', 'count'),
    pct_above_mmr    = ('sold_above_mmr', 'mean')
).reset_index()

kpi_by_make['pct_above_mmr'] = kpi_by_make['pct_above_mmr'] * 100
kpi_by_make = kpi_by_make.sort_values('transaction_count', ascending=False)

output_path = os.path.join(PROCESSED_DIR, 'kpi_by_make.csv')
kpi_by_make.to_csv(output_path, index=False)

print(f"kpi_by_make.csv saved — {len(kpi_by_make)} makes")
print("\nTop 10 Makes by Transaction Count:")
print(kpi_by_make.head(10).to_string(index=False))

kpi_by_make.csv saved — 66 makes

Top 10 Makes by Transaction Count:
     make   avg_price     avg_mmr  avg_deviation_pct  transaction_count  pct_above_mmr
     Ford 14,483.5873 14,676.1825            -0.7581              93976        45.6372
Chevrolet 11,892.7714 12,054.7146            -0.6953              60565        47.4812
   Nissan 11,701.5247 11,827.5546            -0.8002              54015        47.2850
   Toyota 12,233.7897 12,341.9289            -0.4727              39956        46.5712
    Dodge 11,164.7924 11,373.6952            -1.8440              30952        46.2943
    Honda 10,906.5793 10,967.6120             0.1885              27345        49.8300
  Hyundai 11,002.2773 11,224.0205            -1.5533              21836        47.6736
      Bmw 20,861.5646 20,950.3535            -0.1992              20790        49.9086
      Kia 11,807.2807 11,945.8426            -1.1330              18082        47.0302
 Chrysler 11,069.3384 11,316.6619            -1.9328         

## 4. State-Level Aggregation (for Tableau — Regional Pricing View)

In [5]:
kpi_by_state = df.groupby('state').agg(
    avg_price        = ('sellingprice', 'mean'),
    avg_mmr          = ('mmr', 'mean'),
    avg_deviation_pct= ('price_deviation_pct', 'mean'),
    transaction_count= ('sellingprice', 'count'),
    pct_above_mmr    = ('sold_above_mmr', 'mean')
).reset_index()

kpi_by_state['pct_above_mmr'] = kpi_by_state['pct_above_mmr'] * 100
kpi_by_state = kpi_by_state.sort_values('transaction_count', ascending=False)

output_path = os.path.join(PROCESSED_DIR, 'kpi_by_state.csv')
kpi_by_state.to_csv(output_path, index=False)

print(f"kpi_by_state.csv saved — {len(kpi_by_state)} states")
print("\nTop 10 States by Transaction Count:")
print(kpi_by_state.head(10).to_string(index=False))

kpi_by_state.csv saved — 64 states

Top 10 States by Transaction Count:
state   avg_price     avg_mmr  avg_deviation_pct  transaction_count  pct_above_mmr
   fl 13,738.4292 13,931.3719            -1.2927              82924        46.4027
   ca 14,184.3563 14,139.9183             1.8505              73120        54.7607
   pa 15,819.8621 15,987.3012            -1.2554              53899        44.9507
   tx 13,139.6819 13,421.8203            -2.8306              45904        46.9610
   ga 12,791.8507 12,874.5175             3.0262              34744        51.5082
   nj 13,486.5019 14,066.9649            -5.2933              27778        35.0745
   il 14,764.1690 14,952.4709            -2.0881              23473        44.8388
   nc  8,652.3390  8,718.6636             5.2532              21844        53.9507
   oh 14,181.3956 14,380.8886            -2.6614              21567        45.5464
   tn 16,946.9349 16,855.9581             0.6667              20894        53.9389


## 5. Condition Tier Aggregation (for Tableau — Condition Pricing View)

In [6]:
kpi_by_condition = df.groupby('condition_tier').agg(
    avg_price        = ('sellingprice', 'mean'),
    avg_mmr          = ('mmr', 'mean'),
    avg_deviation_pct= ('price_deviation_pct', 'mean'),
    transaction_count= ('sellingprice', 'count'),
    pct_above_mmr    = ('sold_above_mmr', 'mean'),
    avg_condition    = ('condition', 'mean')
).reset_index()

kpi_by_condition['pct_above_mmr'] = kpi_by_condition['pct_above_mmr'] * 100

# Order by condition tier
tier_order = ['Poor', 'Fair', 'Good', 'Excellent']
kpi_by_condition['condition_tier'] = pd.Categorical(
    kpi_by_condition['condition_tier'], categories=tier_order, ordered=True
)
kpi_by_condition = kpi_by_condition.sort_values('condition_tier')

output_path = os.path.join(PROCESSED_DIR, 'kpi_by_condition.csv')
kpi_by_condition.to_csv(output_path, index=False)

print(f"kpi_by_condition.csv saved — {len(kpi_by_condition)} condition tiers")
print(kpi_by_condition.to_string(index=False))

kpi_by_condition.csv saved — 4 condition tiers
condition_tier   avg_price     avg_mmr  avg_deviation_pct  transaction_count  pct_above_mmr  avg_condition
          Poor 12,998.8527 13,620.3581            -7.7354              68219        39.0624         3.1998
          Fair  6,733.1449  7,604.1439            -8.7596              84869        25.6690        21.1632
          Good 10,957.9163 11,391.1666            -0.7959             157220        41.5227        31.2484
     Excellent 17,543.4780 17,265.9435             2.9733             248367        59.4548        41.7652


## 6. Monthly Trend Aggregation (for Tableau — Temporal View)

In [7]:
kpi_monthly_trend = df.groupby(['sale_year', 'sale_month']).agg(
    transaction_count  = ('sellingprice', 'count'),
    total_volume       = ('sellingprice', 'sum'),
    avg_price          = ('sellingprice', 'mean'),
    avg_mmr            = ('mmr', 'mean'),
    avg_deviation_pct  = ('price_deviation_pct', 'mean'),
    pct_above_mmr      = ('sold_above_mmr', 'mean')
).reset_index()

kpi_monthly_trend['pct_above_mmr'] = kpi_monthly_trend['pct_above_mmr'] * 100
kpi_monthly_trend = kpi_monthly_trend.sort_values(['sale_year', 'sale_month'])

# Add month name for readability
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
kpi_monthly_trend['month_label'] = (
    kpi_monthly_trend['sale_month'].map(month_names) + ' ' +
    kpi_monthly_trend['sale_year'].astype(str)
)

output_path = os.path.join(PROCESSED_DIR, 'kpi_monthly_trend.csv')
kpi_monthly_trend.to_csv(output_path, index=False)

print(f"kpi_monthly_trend.csv saved — {len(kpi_monthly_trend)} month-year periods")
print(kpi_monthly_trend[['month_label','transaction_count','avg_price','pct_above_mmr']].to_string(index=False))

kpi_monthly_trend.csv saved — 10 month-year periods
month_label  transaction_count   avg_price  pct_above_mmr
 Jan 2014.0                205 15,625.7317        57.0732
 Feb 2014.0                  1 10,500.0000         0.0000
 Dec 2014.0              53440 11,205.2613        44.5097
 Jan 2015.0             140582 13,202.4145        45.7761
 Feb 2015.0             163026 13,503.3229        48.8388
 Mar 2015.0              46272 13,380.8463        50.8407
 Apr 2015.0               1449 10,126.9876        24.5687
 May 2015.0              52442 14,171.0338        41.4153
 Jun 2015.0              99923 14,808.1131        47.3194
 Jul 2015.0               1297 16,673.4880        47.9568


## 7. Body Type Aggregation (for Tableau — Vehicle Segment View)

In [8]:
kpi_by_body = df.groupby('body').agg(
    avg_price        = ('sellingprice', 'mean'),
    avg_mmr          = ('mmr', 'mean'),
    avg_deviation_pct= ('price_deviation_pct', 'mean'),
    transaction_count= ('sellingprice', 'count'),
    pct_above_mmr    = ('sold_above_mmr', 'mean'),
    avg_odometer     = ('odometer', 'mean'),
    avg_vehicle_age  = ('vehicle_age', 'mean')
).reset_index()

kpi_by_body['pct_above_mmr'] = kpi_by_body['pct_above_mmr'] * 100
kpi_by_body = kpi_by_body.sort_values('transaction_count', ascending=False)

output_path = os.path.join(PROCESSED_DIR, 'kpi_by_body.csv')
kpi_by_body.to_csv(output_path, index=False)

print(f"kpi_by_body.csv saved — {len(kpi_by_body)} body types")
print("\nTop 15 Body Types by Transaction Count:")
print(kpi_by_body.head(15).to_string(index=False))

kpi_by_body.csv saved — 46 body types

Top 15 Body Types by Transaction Count:
        body   avg_price     avg_mmr  avg_deviation_pct  transaction_count  pct_above_mmr  avg_odometer  avg_vehicle_age
       Sedan 11,632.6448 11,801.3655            -1.1078             241310        46.3727   63,127.9996           4.5486
         Suv 15,983.3153 16,116.5938            -0.5754             143826        48.0164   72,209.1362           4.9715
   Hatchback 10,033.8534 10,188.2347            -1.2368              26234        47.0268   52,052.5067           3.8836
     Minivan 11,571.3772 11,696.8124             0.0244              25521        45.6839   75,216.7975           4.5671
       Coupe 15,131.8493 15,202.2241            -0.3062              17749        49.7831   67,765.0960           5.8039
    Crew Cab 21,608.4740 21,717.4369             0.4111              16392        49.3106   79,106.2015           4.9420
       Wagon 10,126.7216 10,251.2078            -1.4011              16124

## 8. Dashboard Data Summary

| File | Rows | Purpose |
|------|------|--------|
| car_prices_cleaned.csv | 558,675 | Full transaction-level data |
| kpi_by_make.csv | ~66 | Brand performance dashboard |
| kpi_by_state.csv | ~50 | Regional pricing map |
| kpi_by_condition.csv | 4 | Condition tier analysis |
| kpi_monthly_trend.csv | ~24 | Temporal trend chart |
| kpi_by_body.csv | ~46 | Vehicle segment comparison |

In [9]:
# Verify all output files exist
output_files = [
    'car_prices_cleaned.csv',
    'kpi_by_make.csv',
    'kpi_by_state.csv',
    'kpi_by_condition.csv',
    'kpi_monthly_trend.csv',
    'kpi_by_body.csv'
]

print("=" * 60)
print("OUTPUT FILE VERIFICATION")
print("=" * 60)
for fname in output_files:
    fpath = os.path.join(PROCESSED_DIR, fname)
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  [OK] {fname:<35} {size_kb:>8.1f} KB")
    else:
        print(f"  [--] {fname:<35} (not yet generated)")

print()
print("✅ All KPI aggregation files exported to data/processed/. Load into Tableau Public for dashboard build.")

OUTPUT FILE VERIFICATION
  [OK] car_prices_cleaned.csv              121027.5 KB
  [OK] kpi_by_make.csv                          4.9 KB
  [OK] kpi_by_state.csv                         4.3 KB
  [OK] kpi_by_condition.csv                     0.5 KB
  [OK] kpi_monthly_trend.csv                    1.2 KB
  [OK] kpi_by_body.csv                          5.3 KB

✅ All KPI aggregation files exported to data/processed/. Load into Tableau Public for dashboard build.
